In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (Unix seconds -> UTC)
metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
trace_df['timestamp_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
log_df['timestamp_dt'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
error_df['timestamp_dt'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# 1) metric.csv aggregates
metric_grp = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .agg(
        count=('value', 'count'),
        min_timestamp=('timestamp_dt', 'min'),
        max_timestamp=('timestamp_dt', 'max'),
        p95_value=('value', lambda x: x.quantile(0.95) if x.notna().any() else np.nan)
    )
    .reset_index()
    .sort_values('count', ascending=False)
)
metric_agg_top = metric_grp.head(20)

metric_overall_range = pd.Series({
    'earliest': metric_df['timestamp_dt'].min(),
    'latest': metric_df['timestamp_dt'].max()
})

# 2) trace.csv aggregates
trace_grp = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .agg(
        count=('value', 'count'),
        min_timestamp=('timestamp_dt', 'min'),
        max_timestamp=('timestamp_dt', 'max'),
        p95_value=('value', lambda x: x.quantile(0.95) if x.notna().any() else np.nan)
    )
    .reset_index()
    .sort_values('count', ascending=False)
)
trace_agg_top = trace_grp.head(20)

trace_overall_range = pd.Series({
    'earliest': trace_df['timestamp_dt'].min(),
    'latest': trace_df['timestamp_dt'].max()
})

# 3) log.csv aggregates
log_grp = (
    log_df
    .groupby(['cmdb_id', 'log_name'], dropna=False)
    .agg(
        count=('value', 'count'),
        min_timestamp=('timestamp_dt', 'min'),
        max_timestamp=('timestamp_dt', 'max'),
        p95_value=('value', lambda x: x.quantile(0.95) if x.notna().any() else np.nan)
    )
    .reset_index()
    .sort_values('count', ascending=False)
)
log_agg_top = log_grp.head(20)

log_overall_range = pd.Series({
    'earliest': log_df['timestamp_dt'].min(),
    'latest': log_df['timestamp_dt'].max()
})

# 4) error_logs.csv aggregates by cmdb_id
error_grp = (
    error_df
    .groupby('cmdb_id', dropna=False)
    .agg(
        count=('message', 'count'),
        min_timestamp=('timestamp_dt', 'min'),
        max_timestamp=('timestamp_dt', 'max')
    )
    .reset_index()
    .sort_values('count', ascending=False)
)
error_agg_top = error_grp.head(20)

error_overall_range = pd.Series({
    'earliest': error_df['timestamp_dt'].min(),
    'latest': error_df['timestamp_dt'].max()
})

# Keep outputs compact: show only top summaries and overall ranges
metric_agg_top, metric_overall_range, trace_agg_top, trace_overall_range, log_agg_top, log_overall_range, error_agg_top, error_overall_range

```
Out[1]:
```
```python
# Build a plain-English summary from the previously computed aggregates.
# Reuse existing variables: metric_agg_top, trace_agg_top, log_agg_top, error_agg_top

def get_p95(df, match_col, cmdb, name_sub):
    try:
        row = df[(df['cmdb_id']==cmdb) & (df[match_col].str.contains(name_sub, na=False))]
        if not row.empty:
            return float(row.iloc[0]['p95_value'])
    except Exception:
        pass
    return None

# Extract key numbers for ts-travel2-service
cmdb = 'ts-travel2-service'
travel_mem_p95 = get_p95(metric_agg_top, 'kpi_name', cmdb, 'mem')
travel_cpu_p95 = get_p95(metric_agg_top, 'kpi_name', cmdb, 'cpu')
travel_diskio_p95 = get_p95(metric_agg_top, 'kpi_name', cmdb, 'diskio')

travel_trace_rowcount_p95 = get_p95(trace_agg_top, 'trace_name', cmdb, 'row_count')
travel_log_rowcount_p95 = get_p95(log_agg_top, 'log_name', cmdb, 'row_count')
travel_log_error_p95 = get_p95(log_agg_top, 'log_name', cmdb, 'error_count')

# Check error logs presence
error_logs_present = not error_agg_top.empty

summary = (
    "Summary of findings:\n"
    "- Likely root-cause service/component: ts-travel2-service.\n"
    "- Reason: High request/row counts indicating a traffic surge or overload:\n"
    f"  * trace row_count p95 = {travel_trace_rowcount_p95 if travel_trace_rowcount_p95 is not None else 'N/A'}\n"
    f"  * log row_count p95   = {travel_log_rowcount_p95 if travel_log_rowcount_p95 is not None else 'N/A'}\n"
    f"  * log error_count p95 = {travel_log_error_p95 if travel_log_error_p95 is not None else 'N/A'} (error counts appear zero in aggregates)\n"
    "- Resource metrics consistent with load-induced pressure on ts-travel2-service:\n"
    f"  * memory p95 = {travel_mem_p95 if travel_mem_p95 is not None else 'N/A'}\n"
    f"  * cpu p95    = {travel_cpu_p95 if travel_cpu_p95 is not None else 'N/A'}\n"
    f"  * diskio p95 = {travel_diskio_p95 if travel_diskio_p95 is not None else 'N/A'}\n"
    "- Error log file: {'contains entries' if error_logs_present else 'no entries (empty)'}; overall log.error_count p95 values are 0.0 for many services, so there is no strong evidence of internal application errors.\n\n"
    "Conclusion: The telemetry points to ts-travel2-service being overloaded by high traffic/row counts, causing elevated resource usage (memory/CPU/disk I/O). This looks like a capacity or traffic surge issue rather than an application error.\n"
    "Suggested next steps: investigate traffic sources to ts-travel2-service, consider scaling the service or applying rate limiting, and continue monitoring resource and latency metrics."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                         cmdb_id    kpi_name  count             min_timestamp             max_timestamp     p95_value
364           ts-voucher-service      socket     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00  4.000000e+00
0    ts-admin-basic-info-service         cpu     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00  5.232017e-01
1    ts-admin-basic-info-service      diskio     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00  5.824456e+04
2    ts-admin-basic-info-service  latency-50     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00  1.762065e-02
3    ts-admin-basic-info-service  latency-90     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00  2.554678e-02
4    ts-admin-basic-info-service         mem     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00  2.303040e+08
5    ts-admin-basic-info-service      socket     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00  7.880000e+00
6    ts-admin-basic-info-service    workload     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00  7.152933e-01
7         ts-admin-order-service         cpu     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00  1.203174e-01
8         ts-admin-order-service      diskio     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00  4.986002e+04
9         ts-admin-order-service         mem     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00  1.922389e+08
10        ts-admin-order-service      socket     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00  3.240000e+00
11        ts-admin-route-service         cpu     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00  1.271723e-01
325             ts-travel2-mongo      socket     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00  1.000000e+01
326           ts-travel2-service         cpu     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00  5.849804e+00
327           ts-travel2-service      diskio     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00  6.235312e+04
328           ts-travel2-service       error     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00  7.093333e-03
329           ts-travel2-service  latency-50     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00  2.267429e-02
330           ts-travel2-service  latency-90     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00  6.693240e-01
331           ts-travel2-service         mem     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00  2.786370e+08, earliest   2024-01-22 15:29:00+00:00
latest     2024-01-22 15:53:00+00:00
dtype: datetime64[ns, UTC],                 cmdb_id                                         trace_name  count             min_timestamp             max_timestamp    p95_value
15                 root                 trace.to_ts-auth-service.row_count     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00    86.000000
14                 root                trace.to_ts-auth-service.error_rate     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00     0.000000
0                  root  trace.to_ts-admin-basic-info-service.duration_...     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00     0.011281
1                  root  trace.to_ts-admin-basic-info-service.duration_p95     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00     0.013923
2                  root    trace.to_ts-admin-basic-info-service.error_rate     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00     0.000000
3                  root     trace.to_ts-admin-basic-info-service.row_count     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00    21.000000
4                  root     trace.to_ts-admin-travel-service.duration_mean     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00     0.480949
5                  root      trace.to_ts-admin-travel-service.duration_p95     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00     0.622077
6                  root        trace.to_ts-admin-travel-service.error_rate     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00     0.000000
736  ts-travel2-service      trace.to_ts-order-other-service.duration_mean     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00     0.007478
737  ts-travel2-service       trace.to_ts-order-other-service.duration_p95     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00     0.014797
738  ts-travel2-service         trace.to_ts-order-other-service.error_rate     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00     0.000000
739  ts-travel2-service          trace.to_ts-order-other-service.row_count     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00    87.600000
740  ts-travel2-service            trace.to_ts-route-service.duration_mean     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00     0.021201
741  ts-travel2-service             trace.to_ts-route-service.duration_p95     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00     0.076136
759  ts-travel2-service              trace.to_ts-travel2-service.row_count     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00  2907.600000
744  ts-travel2-service             trace.to_ts-seat-service.duration_mean     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00     0.062839
745  ts-travel2-service              trace.to_ts-seat-service.duration_p95     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00     0.108084
746  ts-travel2-service                trace.to_ts-seat-service.error_rate     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00     0.000000
747  ts-travel2-service                 trace.to_ts-seat-service.row_count     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00   175.200000, earliest   2024-01-22 15:29:00+00:00
latest     2024-01-22 15:53:00+00:00
dtype: datetime64[ns, UTC],                         cmdb_id         log_name  count             min_timestamp             max_timestamp  p95_value
0   ts-admin-basic-info-service  log.error_count     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00        0.0
1   ts-admin-basic-info-service    log.row_count     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00       31.8
2       ts-admin-travel-service  log.error_count     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00        0.0
3       ts-admin-travel-service    log.row_count     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00       33.0
12             ts-basic-service  log.error_count     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00        0.0
13             ts-basic-service    log.row_count     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00     2694.0
11              ts-auth-service    log.row_count     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00      347.2
10              ts-auth-service  log.error_count     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00        0.0
28          ts-food-map-service  log.error_count     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00        0.0
29          ts-food-map-service    log.row_count     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00       29.2
44       ts-order-other-service  log.error_count     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00        0.0
45       ts-order-other-service    log.row_count     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00      523.6
91              ts-ui-dashboard    log.row_count     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00      289.0
90              ts-ui-dashboard  log.error_count     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00        0.0
65             ts-route-service    log.row_count     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00     1115.0
64             ts-route-service  log.error_count     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00        0.0
67              ts-seat-service    log.row_count     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00     2030.2
66              ts-seat-service  log.error_count     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00        0.0
89           ts-travel2-service    log.row_count     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00     2523.8
88           ts-travel2-service  log.error_count     25 2024-01-22 15:29:00+00:00 2024-01-22 15:53:00+00:00        0.0, earliest   2024-01-22 15:29:00+00:00
latest     2024-01-22 15:53:00+00:00
dtype: datetime64[ns, UTC], Empty DataFrame
Columns: [cmdb_id, count, min_timestamp, max_timestamp]
Index: [], earliest   NaT
latest     NaT
dtype: datetime64[ns])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse metric_df if present, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Ensure timestamp_dt exists and is UTC datetime
if 'timestamp_dt' not in metric_df.columns:
    metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Components of interest
components = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']

# 1) Compute global P95 per (cmdb_id, kpi_name) for these components using full series (before any filtering)
metric_sub = metric_df[metric_df['cmdb_id'].isin(components)].copy()

p95_df = (
    metric_sub
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'p95_value'})
)

# 2) Filter metric.csv to the incident window (inclusive)
window_start = pd.to_datetime("2024-01-22 15:26:57", utc=True)
window_end   = pd.to_datetime("2024-01-22 15:56:57", utc=True)

metric_window = metric_sub[
    (metric_sub['timestamp_dt'] >= window_start) &
    (metric_sub['timestamp_dt'] <= window_end)
].copy()

# 3) For each (cmdb_id, kpi_name) present in the filtered window, compute anomaly_count and earliest_anomaly_timestamp
if not metric_window.empty:
    # attach global p95 to each row by merging
    metric_window = metric_window.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')

    # flag anomalies where value >= global p95 (use >= per instruction)
    metric_window['is_anomaly'] = metric_window['value'] >= metric_window['p95_value']

    agg = (
        metric_window
        .groupby(['cmdb_id', 'kpi_name'], dropna=False)
        .agg(
            p95_value=('p95_value', 'first'),
            total_points_in_window=('value', 'count'),
            anomaly_count=('is_anomaly', 'sum'),
            earliest_anomaly_timestamp=('timestamp_dt', lambda s: s[metric_window.loc[s.index, 'is_anomaly']].min() if metric_window.loc[s.index, 'is_anomaly'].any() else pd.NaT)
        )
        .reset_index()
    )

    # Ensure earliest_anomaly_timestamp is UTC datetime dtype (it will be)
    # Sort to bring most severe / relevant rows to top
    result_df = agg.sort_values(['anomaly_count', 'total_points_in_window'], ascending=[False, False]).head(50)
else:
    # No data in window for these components
    result_df = pd.DataFrame(columns=['cmdb_id','kpi_name','p95_value','total_points_in_window','anomaly_count','earliest_anomaly_timestamp'])

# 4) Distinct list of kpi_name values seen for these components (compact)
distinct_kpis = pd.Series(sorted(p95_df['kpi_name'].unique()), name='kpi_name')

# Display compact outputs: result table (top 50) and distinct kpi list
result_df, distinct_kpis

```
Out[2]:
```
```python
# Build a concise plain-English summary from the previously computed result_df and distinct_kpis.
# Reuse variables in kernel state.

try:
    _ = result_df
    _ = distinct_kpis
except NameError:
    raise RuntimeError("Required results (result_df, distinct_kpis) not found in kernel state. Please re-run the computation step.")

# Ensure correct dtypes
result_df = result_df.copy()
if 'earliest_anomaly_timestamp' in result_df.columns:
    result_df['earliest_anomaly_timestamp'] = pd.to_datetime(result_df['earliest_anomaly_timestamp'], utc=True)

# Identify fully anomalous series (anomaly_count == total_points_in_window)
fully_anom = result_df[result_df['anomaly_count'] == result_df['total_points_in_window']]

# Top absolute anomaly by anomaly_count
top_abs = result_df.sort_values('anomaly_count', ascending=False).head(1)

def fmt_row(row):
    ts = row['earliest_anomaly_timestamp']
    ts_str = ts.isoformat() if pd.notna(ts) else 'N/A'
    return f"{row['cmdb_id']} - {row['kpi_name']}: {int(row['anomaly_count'])}/{int(row['total_points_in_window'])} anomalies, first at {ts_str}"

parts = []
parts.append("Summary of anomaly analysis for selected components (window 2024-01-22 15:26:57 to 2024-01-22 15:56:57 UTC):")

if not fully_anom.empty:
    parts.append("- Series with anomalies at all points in the window:")
    for _, r in fully_anom.iterrows():
        parts.append("  * " + fmt_row(r))
else:
    parts.append("- No series had anomalies for every point in the window.")

if not top_abs.empty:
    r = top_abs.iloc[0]
    parts.append("- Series with the highest absolute anomaly count:")
    parts.append("  * " + fmt_row(r))

# Provide a short list of other notable anomalies (top 5 by anomaly_count)
others = result_df.sort_values(['anomaly_count', 'total_points_in_window'], ascending=[False, False]).head(5)
parts.append("- Top anomaly series (up to 5):")
for _, r in others.iterrows():
    parts.append("  * " + fmt_row(r))

# Distinct KPI names seen
kpi_list = ", ".join(distinct_kpis.tolist())
parts.append(f"- Distinct kpi_name values observed for these components: {kpi_list}.")

parts.append("\nKey takeaways:")
parts.append("- ts-route-service and ts-train-service show error series fully anomalous in the window (all points >= global P95).")
parts.append("- ts-travel-service has multiple metrics with anomalies (socket, mem, latency, etc.), but typically only a few points exceed global P95.")
parts.append("- ts-auth-service shows anomalies on cpu and diskio (a small number of points exceed global P95).")
parts.append("- Investigate ts-route-service and ts-train-service errors first (persistent anomalies), then inspect load/latency on ts-travel-service and resource spikes on ts-auth-service.")

summary = "\n".join(parts)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name     p95_value  total_points_in_window  anomaly_count earliest_anomaly_timestamp
16   ts-route-service       error  0.000000e+00                      11             11  2024-01-22 15:43:00+00:00
36  ts-travel-service      socket  1.785000e+01                      25              3  2024-01-22 15:29:00+00:00
24   ts-train-service       error  0.000000e+00                       3              3  2024-01-22 15:51:00+00:00
0     ts-auth-service         cpu  1.557666e+01                      25              2  2024-01-22 15:44:00+00:00
1     ts-auth-service      diskio  5.735977e+04                      25              2  2024-01-22 15:31:00+00:00
..                ...         ...           ...                     ...            ...                        ...
32  ts-travel-service       error  3.573333e-03                      25              2  2024-01-22 15:42:00+00:00
33  ts-travel-service  latency-50  2.634631e-02                      25              2  2024-01-22 15:45:00+00:00
34  ts-travel-service  latency-90  3.946592e-01                      25              2  2024-01-22 15:43:00+00:00
35  ts-travel-service         mem  2.701898e+08                      25              2  2024-01-22 15:51:00+00:00
37  ts-travel-service    workload  1.046373e+01                      25              2  2024-01-22 15:39:00+00:00

[38 rows x 6 columns], 0           cpu
1        diskio
2         error
3    latency-50
4    latency-90
5           mem
6        socket
7      workload
Name: kpi_name, dtype: object)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse trace_df if available otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure timestamp_dt parsed as UTC datetimes
if 'timestamp_dt' not in trace_df.columns:
    trace_df['timestamp_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Services of interest (match substring patterns in trace_name)
patterns = [
    'to_ts-auth-service',
    'to_ts-order-service',
    'to_ts-route-service',
    'to_ts-train-service',
    'to_ts-travel-service'
]
pattern_regex = "|".join(patterns)

# Subset trace rows that reference these services in trace_name
trace_sub_all = trace_df[trace_df['trace_name'].str.contains(pattern_regex, na=False)]

# 1) Compute global P95 per (cmdb_id, trace_name) using full available series
p95_trace = (
    trace_sub_all
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'p95_value'})
)

# 2) Filter to incident window (inclusive)
window_start = pd.to_datetime("2024-01-22 15:26:57", utc=True)
window_end   = pd.to_datetime("2024-01-22 15:56:57", utc=True)

trace_window = trace_sub_all[
    (trace_sub_all['timestamp_dt'] >= window_start) &
    (trace_sub_all['timestamp_dt'] <= window_end)
].copy()

# 3) For each (cmdb_id, trace_name) present in the window compute metrics
if not trace_window.empty:
    # merge p95
    trace_window = trace_window.merge(p95_trace, on=['cmdb_id', 'trace_name'], how='left')
    # flag anomalies where value >= global p95
    trace_window['is_anomaly'] = trace_window['value'] >= trace_window['p95_value']
    # aggregate
    agg = (
        trace_window
        .groupby(['cmdb_id', 'trace_name'], dropna=False)
        .agg(
            p95_value=('p95_value', 'first'),
            total_points_in_window=('value', 'count'),
            anomaly_count=('is_anomaly', 'sum'),
            earliest_anomaly_timestamp=('timestamp_dt', lambda x: x[trace_window.loc[x.index, 'is_anomaly']].min() if trace_window.loc[x.index, 'is_anomaly'].any() else pd.NaT)
        )
        .reset_index()
    )
    # sort by anomaly_count desc then earliest_anomaly_timestamp asc
    agg_sorted = agg.sort_values(['anomaly_count', 'earliest_anomaly_timestamp'], ascending=[False, True])
    result_df = agg_sorted[['cmdb_id', 'trace_name', 'p95_value', 'total_points_in_window', 'anomaly_count', 'earliest_anomaly_timestamp']].head(20)
    # 4) short list of trace_names that had anomaly_count > 0 (distinct)
    anomaly_trace_names = pd.Series(sorted(result_df[result_df['anomaly_count'] > 0]['trace_name'].unique()), name='trace_name')
else:
    # empty results if no data in window
    result_df = pd.DataFrame(columns=['cmdb_id', 'trace_name', 'p95_value', 'total_points_in_window', 'anomaly_count', 'earliest_anomaly_timestamp'])
    anomaly_trace_names = pd.Series(dtype=object, name='trace_name')

# Display compact results
result_df, anomaly_trace_names

```
Out[3]:
```
```python
# Compose a plain-English summary using existing results (result_df, anomaly_trace_names).
try:
    _ = result_df
    _ = anomaly_trace_names
except NameError:
    raise RuntimeError("Expected variables (result_df, anomaly_trace_names) not found. Please re-run the analysis cell first.")

# Build summary
summary = (
    "Summary of trace analysis for services referencing to_ts-<service> (window 2024-01-22 15:26:57 to 15:56:57 UTC):\n\n"
    "Key findings:\n"
    "- The only trace metric showing 'anomalies' for these services is the error_rate traces (e.g., trace.to_ts-auth-service.error_rate, trace.to_ts-order-service.error_rate, trace.to_ts-route-service.error_rate, trace.to_ts-train-service.error_rate, trace.to_ts-travel-service.error_rate).\n"
    "- For those trace.error_rate series the computed global P95 is 0.0. Because we mark anomalies as value >= global P95, every sampled point in the incident window meets that condition.\n"
    "  * As a result anomaly_count equals total_points_in_window (typically 24–25) and the earliest_anomaly_timestamp is the first sample in the window (e.g. 2024-01-22T15:29:00+00:00) for many series.\n\n"
    "Interpretation and recommendation:\n"
    "- This pattern (global P95 == 0 and thus all points flagged) is likely a threshold artifact, not proof of active errors. It produces false-positive anomalies because the threshold is zero.\n"
    "- Prioritize investigating non-error_rate traces/metrics (for example latency, row_count, resource KPIs) for real performance issues. Specifically, check latency and row_count traces and metric KPIs for spikes, and inspect services that showed elevated latency or row_count in earlier metric analysis.\n\n"
    "Distinct trace names that had anomaly_count > 0 (from this analysis):\n  - " + "\n  - ".join(anomaly_trace_names.tolist()) + "\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                      cmdb_id                             trace_name  p95_value  total_points_in_window  anomaly_count earliest_anomaly_timestamp
2                        root    trace.to_ts-auth-service.error_rate        0.0                      25             25  2024-01-22 15:29:00+00:00
10                       root  trace.to_ts-travel-service.error_rate        0.0                      25             25  2024-01-22 15:29:00+00:00
14    ts-admin-travel-service  trace.to_ts-travel-service.error_rate        0.0                      25             25  2024-01-22 15:29:00+00:00
18            ts-auth-service    trace.to_ts-auth-service.error_rate        0.0                      25             25  2024-01-22 15:29:00+00:00
22           ts-basic-service   trace.to_ts-route-service.error_rate        0.0                      25             25  2024-01-22 15:29:00+00:00
26           ts-basic-service   trace.to_ts-train-service.error_rate        0.0                      25             25  2024-01-22 15:29:00+00:00
30            ts-food-service  trace.to_ts-travel-service.error_rate        0.0                      25             25  2024-01-22 15:29:00+00:00
38           ts-order-service   trace.to_ts-order-service.error_rate        0.0                      25             25  2024-01-22 15:29:00+00:00
50           ts-route-service   trace.to_ts-route-service.error_rate        0.0                      25             25  2024-01-22 15:29:00+00:00
54            ts-seat-service   trace.to_ts-order-service.error_rate        0.0                      25             25  2024-01-22 15:29:00+00:00
58            ts-seat-service  trace.to_ts-travel-service.error_rate        0.0                      25             25  2024-01-22 15:29:00+00:00
66           ts-train-service   trace.to_ts-train-service.error_rate        0.0                      25             25  2024-01-22 15:29:00+00:00
70          ts-travel-service   trace.to_ts-order-service.error_rate        0.0                      25             25  2024-01-22 15:29:00+00:00
74          ts-travel-service   trace.to_ts-route-service.error_rate        0.0                      25             25  2024-01-22 15:29:00+00:00
78          ts-travel-service   trace.to_ts-train-service.error_rate        0.0                      25             25  2024-01-22 15:29:00+00:00
82          ts-travel-service  trace.to_ts-travel-service.error_rate        0.0                      25             25  2024-01-22 15:29:00+00:00
86         ts-travel2-service   trace.to_ts-route-service.error_rate        0.0                      25             25  2024-01-22 15:29:00+00:00
90         ts-travel2-service   trace.to_ts-train-service.error_rate        0.0                      25             25  2024-01-22 15:29:00+00:00
6                        root   trace.to_ts-order-service.error_rate        0.0                      24             24  2024-01-22 15:30:00+00:00
34  ts-inside-payment-service   trace.to_ts-order-service.error_rate        0.0                      24             24  2024-01-22 15:30:00+00:00, 0      trace.to_ts-auth-service.error_rate
1     trace.to_ts-order-service.error_rate
2     trace.to_ts-route-service.error_rate
3     trace.to_ts-train-service.error_rate
4    trace.to_ts-travel-service.error_rate
Name: trace_name, dtype: object)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse loaded dataframes if available, otherwise load
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")
if 'timestamp_dt' not in log_df.columns:
    log_df['timestamp_dt'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

try:
    error_df
except NameError:
    error_df = pd.read_csv("error_logs.csv")
if 'timestamp_dt' not in error_df.columns:
    error_df['timestamp_dt'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# Components of interest
components = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']

# 1) Compute global P95 per (cmdb_id, log_name) using full series for these components
log_sub = log_df[log_df['cmdb_id'].isin(components)].copy()
p95_log = (
    log_sub
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'p95_value'})
)

# 2) Filter log.csv to the incident window (inclusive)
window_start = pd.to_datetime("2024-01-22 15:26:57", utc=True)
window_end   = pd.to_datetime("2024-01-22 15:56:57", utc=True)

log_window = log_sub[
    (log_sub['timestamp_dt'] >= window_start) &
    (log_sub['timestamp_dt'] <= window_end)
].copy()

# 3) For each (cmdb_id, log_name) present in the filtered window compute aggregates
if not log_window.empty:
    # merge p95 (global) into windowed rows
    log_window = log_window.merge(p95_log, on=['cmdb_id', 'log_name'], how='left')
    # flag anomalies where value >= global p95 (use >= per instruction)
    log_window['is_anomaly'] = log_window['value'] >= log_window['p95_value']
    # aggregate per series
    log_agg = (
        log_window
        .groupby(['cmdb_id', 'log_name'], dropna=False)
        .apply(lambda g: pd.Series({
            'p95_value': g['p95_value'].iloc[0],
            'total_points_in_window': int(g['value'].count()),
            'anomaly_count': int(g['is_anomaly'].sum()),
            'earliest_anomaly_timestamp': (g.loc[g['is_anomaly'], 'timestamp_dt'].min() if g['is_anomaly'].any() else pd.NaT)
        }))
        .reset_index()
    )
    # sort by anomaly_count desc then earliest_anomaly_timestamp asc
    log_result_df = log_agg.sort_values(['anomaly_count', 'earliest_anomaly_timestamp'], ascending=[False, True]).head(20)
else:
    # empty result structure
    log_result_df = pd.DataFrame(columns=['cmdb_id','log_name','p95_value','total_points_in_window','anomaly_count','earliest_anomaly_timestamp'])

# Distinct log_name values seen for these components (compact)
distinct_log_names = pd.Series(sorted(p95_log['log_name'].unique()), name='log_name')

# --- error_logs.csv analysis for same components and window ---
err_sub = error_df[error_df['cmdb_id'].isin(components)].copy()
err_window = err_sub[
    (err_sub['timestamp_dt'] >= window_start) &
    (err_sub['timestamp_dt'] <= window_end)
].copy()

if not err_window.empty:
    # aggregate per cmdb_id: count, earliest, latest, up to 3 sample messages (truncated)
    def sample_msgs(series):
        # take up to first 3 messages, truncate to 200 chars each
        return list(series.astype(str).head(3).apply(lambda x: x[:200]))
    error_summary_df = (
        err_window
        .groupby('cmdb_id', dropna=False)
        .agg(
            count_of_error_messages = ('message', 'count'),
            earliest_message_timestamp = ('timestamp_dt', 'min'),
            latest_message_timestamp = ('timestamp_dt', 'max'),
            sample_messages = ('message', sample_msgs)
        )
        .reset_index()
        .sort_values('count_of_error_messages', ascending=False)
        .head(10)
    )
else:
    error_summary_df = pd.DataFrame(columns=['cmdb_id','count_of_error_messages','earliest_message_timestamp','latest_message_timestamp','sample_messages'])

# Compact outputs
log_result_df, distinct_log_names, error_summary_df

```
Out[4]:
```
Summary (plain English):

- Distinct log metrics observed for these components: log.error_count, log.row_count.

- log.error_count:
  - Global P95 = 0.0 for all listed services. Because the P95 is zero, every point in the incident window meets value >= P95 and is flagged as an anomaly (25 points each).
  - Earliest flagged timestamp for these series is 2024-01-22T15:29:00Z (first sample in the window). This is a threshold artifact (P95 = 0) and not reliable evidence of active errors.

- log.row_count (more meaningful spikes):
  - ts-travel-service: p95 = 1965.2, 25 points in window, anomaly_count = 2, earliest anomaly at 2024-01-22T15:30:00Z.
  - ts-order-service: p95 = 429.6, 25 points, anomaly_count = 2, earliest anomaly at 2024-01-22T15:30:00Z.
  - ts-auth-service: p95 = 347.2, 25 points, anomaly_count = 2, earliest anomaly at 2024-01-22T15:37:00Z.
  - ts-route-service: p95 = 1115.0, 25 points, anomaly_count = 2, earliest anomaly at 2024-01-22T15:39:00Z.
  - ts-train-service: only 2 points in the window for these logs; log.error_count flagged both points (artifact), log.row_count shows 1 anomaly at 2024-01-22T15:50:00Z.

- error_logs.csv: No error log messages for the selected components in the incident window (empty result).

Interpretation and recommended next steps:
- Ignore the “anomalies” coming from log.error_count where global P95 = 0 (these are false positives due to a zero threshold).
- Focus investigation on the log.row_count spikes (ts-travel-service, ts-order-service, ts-auth-service, ts-route-service) which have a small number of points exceeding their global P95 around ~15:30–15:40 UTC. These indicate brief traffic/load surges that could cause downstream pressure.
- Since no application error messages were found in error_logs for these components during the window, prioritize investigating traffic sources, request patterns, and scaling/capacity for the services with elevated row_count (especially ts-travel-service and ts-order-service).

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id         log_name  p95_value  total_points_in_window  anomaly_count earliest_anomaly_timestamp
0    ts-auth-service  log.error_count        0.0                      25             25  2024-01-22 15:29:00+00:00
2   ts-order-service  log.error_count        0.0                      25             25  2024-01-22 15:29:00+00:00
4   ts-route-service  log.error_count        0.0                      25             25  2024-01-22 15:29:00+00:00
8  ts-travel-service  log.error_count        0.0                      25             25  2024-01-22 15:29:00+00:00
3   ts-order-service    log.row_count      429.6                      25              2  2024-01-22 15:30:00+00:00
9  ts-travel-service    log.row_count     1965.2                      25              2  2024-01-22 15:30:00+00:00
1    ts-auth-service    log.row_count      347.2                      25              2  2024-01-22 15:37:00+00:00
5   ts-route-service    log.row_count     1115.0                      25              2  2024-01-22 15:39:00+00:00
6   ts-train-service  log.error_count        0.0                       2              2  2024-01-22 15:50:00+00:00
7   ts-train-service    log.row_count      153.0                       2              1  2024-01-22 15:50:00+00:00, 0    log.error_count
1      log.row_count
Name: log_name, dtype: object, Empty DataFrame
Columns: [cmdb_id, count_of_error_messages, earliest_message_timestamp, latest_message_timestamp, sample_messages]
Index: [])```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse trace_df if present, otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure timestamp_dt exists and is parsed to UTC
if 'timestamp_dt' not in trace_df.columns:
    trace_df['timestamp_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Target service substring patterns to match in trace_name
patterns = [
    'to_ts-auth-service',
    'to_ts-order-service',
    'to_ts-route-service',
    'to_ts-train-service',
    'to_ts-travel-service'
]
pattern_regex = "|".join(patterns)

# Subset trace rows that reference these services
trace_sub_all = trace_df[trace_df['trace_name'].str.contains(pattern_regex, na=False)].copy()

# 1) Compute global P95 of 'value' per distinct trace_name using full available series
p95_by_trace_name = (
    trace_sub_all
    .groupby('trace_name', dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'p95_value'})
)

# 2) Filter to the incident window (inclusive)
window_start = pd.to_datetime("2024-01-22 15:26:57", utc=True)
window_end   = pd.to_datetime("2024-01-22 15:56:57", utc=True)

trace_window = trace_sub_all[
    (trace_sub_all['timestamp_dt'] >= window_start) &
    (trace_sub_all['timestamp_dt'] <= window_end)
].copy()

# 3) Exclude trace_name entries that end with '.error_rate' and compute aggregates
if not trace_window.empty:
    # Exclude error_rate traces
    trace_window = trace_window[~trace_window['trace_name'].str.endswith('.error_rate', na=False)].copy()
    # Merge global p95 (by trace_name)
    trace_window = trace_window.merge(p95_by_trace_name, on='trace_name', how='left')
    # Flag anomalies where value >= global p95
    trace_window['is_anomaly'] = trace_window['value'] >= trace_window['p95_value']
    # Aggregate per trace_name
    def agg_group(g):
        return pd.Series({
            'p95_value': g['p95_value'].iloc[0],
            'total_points_in_window': int(g.shape[0]),
            'anomaly_count': int(g['is_anomaly'].sum()),
            'earliest_anomaly_timestamp': (g.loc[g['is_anomaly'], 'timestamp_dt'].min() if g['is_anomaly'].any() else pd.NaT)
        })
    agg = trace_window.groupby('trace_name', dropna=False).apply(agg_group).reset_index()
    # Sort by anomaly_count desc then earliest_anomaly_timestamp asc and limit to top 20
    result_df = agg.sort_values(['anomaly_count', 'earliest_anomaly_timestamp'], ascending=[False, True]).head(20)
else:
    # No non-error_rate trace data in window
    result_df = pd.DataFrame(columns=['trace_name','p95_value','total_points_in_window','anomaly_count','earliest_anomaly_timestamp'])

# Distinct list of trace_name values considered (compact)
distinct_trace_names = pd.Series(sorted(trace_sub_all['trace_name'].unique()), name='trace_name')

# Return compact outputs
result_df, distinct_trace_names

```
Out[5]:
```
Summary (plain English):

What I checked
- I looked at traces that reference the five target services (matches like "to_ts-<service>") and excluded any ".error_rate" traces from the anomaly summary.
- Global P95 thresholds were computed from the full trace series. Then the incident window 2024-01-22 15:26:57 → 2024-01-22 15:56:57 UTC was inspected for points >= those P95s.

Key findings (top signals)
- trace.to_ts-order-service.*  
  - row_count: p95 ≈ 421.65, 168 points in window, anomaly_count = 9, earliest anomaly at 2024-01-22T15:30:00Z.  
  - duration_mean: p95 ≈ 0.3628, 168 pts, anomaly_count = 9, earliest anomaly ~ 2024-01-22T15:43:00Z.  
  - duration_p95: p95 ≈ 0.39198, 168 pts, anomaly_count = 9, earliest anomaly ~ 2024-01-22T15:42:00Z.
- trace.to_ts-travel-service.*  
  - row_count: p95 ≈ 1450.50, 146 pts, anomaly_count = 8, earliest anomaly 2024-01-22T15:30:00Z.  
  - duration_mean / duration_p95 also show 8 anomalies with earliest anomalies around 15:31–15:44Z (duration_p95 p95 ≈ 1.2876).
- trace.to_ts-route-service.* and trace.to_ts-train-service.*  
  - row_count: both show p95s large (route ≈ 2038.3, train ≈ 2500.2), 100 pts each, anomaly_count = 5, earliest anomaly 2024-01-22T15:30:00Z.  
  - Their duration_mean/duration_p95 series also show the same anomaly counts with earliest anomalies near 15:42–15:50Z.
- trace.to_ts-auth-service.*  
  - row_count: p95 ≈ 342.2, 50 pts, anomaly_count = 3, earliest anomaly 2024-01-22T15:37:00Z.  
  - duration series show 3 anomalies with earliest around 15:42Z.

Distinct trace_names considered (examples)
- trace.to_ts-auth-service.duration_mean, duration_p95, error_rate, row_count  
- trace.to_ts-order-service.duration_mean, duration_p95, error_rate, row_count  
- trace.to_ts-route-service.duration_mean, duration_p95, error_rate, row_count  
- trace.to_ts-train-service.duration_mean, duration_p95, error_rate, row_count  
- trace.to_ts-travel-service.duration_mean, duration_p95, error_rate, row_count

Interpretation
- The prominent anomalies are in row_count and latency (duration_mean / duration_p95) for multiple target services, with many series showing anomaly points clustered around ~15:30Z and additional latency anomalies around ~15:42–15:50Z.
- This pattern indicates short-lived traffic spikes (higher row_count) that coincide with increased latencies on the affected directional traces (order, travel, route, train, and to a lesser extent auth).
- These are not the zero-threshold error_rate artifacts (error_rate traces were excluded from this summary); the row_count/latency anomalies are meaningful signals of load/latency pressure.

Recommended next steps
- Investigate traffic sources and request patterns around ~15:30Z and ~15:40–15:50Z (e.g., a surge, batch job, or replay).  
- Check scaling/instances, queue/backpressure behavior, and any rate-limiting or downstream contention for ts-order-service and ts-travel-service first (they show the largest counts of anomalies).  
- Correlate with host/resource metrics (CPU/memory/disk I/O) and logs to identify whether resource exhaustion or downstream slow services caused the latency spikes.

The original code execution output of IPython Kernel is also provided below for reference:

(                                  trace_name    p95_value  total_points_in_window  anomaly_count earliest_anomaly_timestamp
5        trace.to_ts-order-service.row_count   421.650000                     168              9  2024-01-22 15:30:00+00:00
4     trace.to_ts-order-service.duration_p95     0.391982                     168              9  2024-01-22 15:42:00+00:00
3    trace.to_ts-order-service.duration_mean     0.362763                     168              9  2024-01-22 15:43:00+00:00
14      trace.to_ts-travel-service.row_count  1450.500000                     146              8  2024-01-22 15:30:00+00:00
13   trace.to_ts-travel-service.duration_p95     1.287592                     146              8  2024-01-22 15:31:00+00:00
12  trace.to_ts-travel-service.duration_mean     0.856912                     146              8  2024-01-22 15:44:00+00:00
8        trace.to_ts-route-service.row_count  2038.300000                     100              5  2024-01-22 15:30:00+00:00
11       trace.to_ts-train-service.row_count  2500.200000                     100              5  2024-01-22 15:30:00+00:00
7     trace.to_ts-route-service.duration_p95     0.075874                     100              5  2024-01-22 15:42:00+00:00
6    trace.to_ts-route-service.duration_mean     0.042553                     100              5  2024-01-22 15:44:00+00:00
9    trace.to_ts-train-service.duration_mean     0.005467                     100              5  2024-01-22 15:50:00+00:00
10    trace.to_ts-train-service.duration_p95     0.007002                     100              5  2024-01-22 15:50:00+00:00
2         trace.to_ts-auth-service.row_count   342.200000                      50              3  2024-01-22 15:37:00+00:00
0     trace.to_ts-auth-service.duration_mean     0.391621                      50              3  2024-01-22 15:42:00+00:00
1      trace.to_ts-auth-service.duration_p95     1.029986                      50              3  2024-01-22 15:42:00+00:00, 0       trace.to_ts-auth-service.duration_mean
1        trace.to_ts-auth-service.duration_p95
2          trace.to_ts-auth-service.error_rate
3           trace.to_ts-auth-service.row_count
4      trace.to_ts-order-service.duration_mean
5       trace.to_ts-order-service.duration_p95
6         trace.to_ts-order-service.error_rate
7          trace.to_ts-order-service.row_count
8      trace.to_ts-route-service.duration_mean
9       trace.to_ts-route-service.duration_p95
10        trace.to_ts-route-service.error_rate
11         trace.to_ts-route-service.row_count
12     trace.to_ts-train-service.duration_mean
13      trace.to_ts-train-service.duration_p95
14        trace.to_ts-train-service.error_rate
15         trace.to_ts-train-service.row_count
16    trace.to_ts-travel-service.duration_mean
17     trace.to_ts-travel-service.duration_p95
18       trace.to_ts-travel-service.error_rate
19        trace.to_ts-travel-service.row_count
Name: trace_name, dtype: object)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import re

# Reuse trace_df if available, otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
    trace_df['timestamp_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Ensure timestamp_dt exists and is UTC
if 'timestamp_dt' not in trace_df.columns:
    trace_df['timestamp_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# 1) Select trace rows where trace_name contains 'to_ts-' and does NOT end with '.error_rate'
mask_to_ts = trace_df['trace_name'].str.contains(r'\bto_ts-', na=False)
mask_not_error = ~trace_df['trace_name'].str.endswith('.error_rate', na=False)
trace_sel = trace_df[mask_to_ts & mask_not_error].copy()

# 1) Compute global P95 per full-series trace_name (using full available series)
p95_by_trace = (
    trace_sel
    .groupby('trace_name', dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'p95_value'})
)

# 2) Filter to incident window (inclusive)
window_start = pd.to_datetime("2024-01-22 15:26:57", utc=True)
window_end   = pd.to_datetime("2024-01-22 15:56:57", utc=True)

trace_window = trace_sel[
    (trace_sel['timestamp_dt'] >= window_start) &
    (trace_sel['timestamp_dt'] <= window_end)
].copy()

# If no data in window, produce empty results
if trace_window.empty:
    per_destination_df = pd.DataFrame(columns=['destination_service','distinct_origin_count','total_breach_count','earliest_breach_overall'])
    top_origin_rows = pd.DataFrame(columns=['origin_cmdb_id','breach_count','earliest_breach'])
else:
    # 3) Extract destination_service from trace_name: substring after 'to_ts-' up to next '.'
    # Example: 'trace.to_ts-order-service.duration_mean' -> 'order-service'
    def extract_dest(s):
        m = re.search(r'to_ts-([^.\s]+)', s)
        return m.group(1) if m else np.nan
    trace_window['destination_service'] = trace_window['trace_name'].apply(extract_dest)

    # Merge global p95 by trace_name into window rows (global P95 computed on full series)
    trace_window = trace_window.merge(p95_by_trace, on='trace_name', how='left')

    # Flag breaches where value >= global P95 (use >= as specified)
    trace_window['is_breach'] = trace_window['value'] >= trace_window['p95_value']

    # 4) Per (origin_cmdb_id = cmdb_id, destination_service, trace_name) compute breach_count and earliest_breach
    per_origin_dest_trace = (
        trace_window
        .groupby(['cmdb_id', 'destination_service', 'trace_name'], dropna=False)
        .agg(
            trace_total_points_in_window=('value', 'count'),
            breach_count_trace=('is_breach', 'sum'),
            earliest_breach_trace=('timestamp_dt', lambda s: s[trace_window.loc[s.index, 'is_breach']].min() if trace_window.loc[s.index, 'is_breach'].any() else pd.NaT)
        )
        .reset_index()
    )

    # 4 cont) For each (origin_cmdb_id, destination_service) compute total_breach_points (sum of breaches across trace_names) and earliest_breach
    per_origin_dest = (
        per_origin_dest_trace
        .groupby(['cmdb_id', 'destination_service'], dropna=False)
        .agg(
            total_breach_points=('breach_count_trace', 'sum'),
            earliest_breach=('earliest_breach_trace', lambda s: s.dropna().min() if s.dropna().size>0 else pd.NaT)
        )
        .reset_index()
    )

    # 5) Aggregate to per-destination rows:
    # distinct_origin_count = number of distinct origin_cmdb_id with total_breach_points > 0 to this destination
    # total_breach_count = sum of total_breach_points across origins
    # earliest_breach_overall = min(earliest_breach across origins)
    per_destination = (
        per_origin_dest
        .assign(has_breach=lambda df: df['total_breach_points'] > 0)
        .groupby('destination_service', dropna=False)
        .agg(
            distinct_origin_count=('has_breach', lambda s: int(s.sum())),
            total_breach_count=('total_breach_points', 'sum'),
            earliest_breach_overall=('earliest_breach', lambda s: s.dropna().min() if s.dropna().size>0 else pd.NaT)
        )
        .reset_index()
    )

    # 6A) Per-destination summary table sorted by total_breach_count desc then earliest_breach_overall asc (limit top 20)
    per_destination_df = per_destination.sort_values(['total_breach_count', 'earliest_breach_overall'], ascending=[False, True]).head(20)

    # 6B) For the top destination_service from A, list up to top 10 origin_cmdb_id rows with breach_count and earliest_breach
    if per_destination_df.shape[0] > 0:
        top_destination = per_destination_df.iloc[0]['destination_service']
        origins_for_top = per_origin_dest[per_origin_dest['destination_service'] == top_destination].copy()
        # keep only origins with breaches (>0) and sort by breach_count desc
        origins_for_top = origins_for_top[origins_for_top['total_breach_points'] > 0]
        top_origin_rows = origins_for_top.sort_values('total_breach_points', ascending=False).head(10)
        top_origin_rows = top_origin_rows.rename(columns={
            'cmdb_id': 'origin_cmdb_id',
            'total_breach_points': 'breach_count',
            'earliest_breach': 'earliest_breach'
        })[['origin_cmdb_id', 'breach_count', 'earliest_breach']]
    else:
        top_origin_rows = pd.DataFrame(columns=['origin_cmdb_id','breach_count','earliest_breach'])

# Return compact outputs
per_destination_df, top_origin_rows

```
Out[6]:
```
```python
# Build a plain-English summary from the existing per_destination_df and top_origin_rows variables.
# Reuse kernel state variables created earlier.

try:
    _ = per_destination_df
    _ = top_origin_rows
except NameError:
    raise RuntimeError("Expected results (per_destination_df, top_origin_rows) not found in kernel state.")

# Prepare brief top-destination list (top 5) for context
top_destinations = per_destination_df.head(5).copy()
top_dest_lines = []
for _, r in top_destinations.iterrows():
    ts = r['earliest_breach_overall']
    ts_str = pd.to_datetime(ts).isoformat() if pd.notna(ts) else 'N/A'
    top_dest_lines.append(f"{r['destination_service']}: total_breach_count={int(r['total_breach_count'])}, distinct_origin_count={int(r['distinct_origin_count'])}, earliest_breach={ts_str}")

# Prepare top origins for the top destination
if not top_origin_rows.empty:
    origin_lines = []
    for _, r in top_origin_rows.iterrows():
        ts = r['earliest_breach']
        ts_str = pd.to_datetime(ts).isoformat() if pd.notna(ts) else 'N/A'
        origin_lines.append(f"{r['origin_cmdb_id']}: breach_count={int(r['breach_count'])}, earliest_breach={ts_str}")
else:
    origin_lines = ["No origin breaches found for top destination."]

summary = (
    "Summary of trace breach analysis (traces with 'to_ts-' excluding '.error_rate', window 2024-01-22 15:26:57 to 15:56:57 UTC):\n\n"
    "Top affected destination services (top 5):\n  - " + "\n  - ".join(top_dest_lines) + "\n\n"
    "Top destination (highest total_breach_count):\n  - " + str(top_destinations.iloc[0]['destination_service']) + "\n\n"
    "Origins contributing breaches to that top destination (up to 10, sorted by breach_count):\n  - " + "\n  - ".join(origin_lines) + "\n\n"
    "Interpretation:\n"
    "- The leading destination is order-service (highest total breaches). Multiple origins (notably 'root' and 'ts-order-service') generated the most breaches to it.\n"
    "- Breaches are concentrated around ~2024-01-22T15:30:00Z (earliest overall) and additional origin-specific earliest times (see origins above).\n"
    "- This pattern indicates a traffic/latency surge toward several downstream services (order-service, travel-service, train/route/seat-related services). Investigate the high-volume origins and request patterns around ~15:30Z, and check scaling/backpressure and downstream latency during that window.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         destination_service  distinct_origin_count  total_breach_count   earliest_breach_overall
13             order-service                      2                  27 2024-01-22 15:30:00+00:00
12       order-other-service                      2                  27 2024-01-22 15:31:00+00:00
21           station-service                      2                  27 2024-01-22 15:36:00+00:00
24            travel-service                      2                  24 2024-01-22 15:30:00+00:00
25           travel2-service                      3                  21 2024-01-22 15:30:00+00:00
19              seat-service                      5                  18 2024-01-22 15:30:00+00:00
22        ticketinfo-service                      3                  18 2024-01-22 15:30:00+00:00
18             route-service                      4                  15 2024-01-22 15:30:00+00:00
23             train-service                      4                  15 2024-01-22 15:30:00+00:00
8           contacts-service                      4                  15 2024-01-22 15:33:00+00:00
10              food-service                      2                  15 2024-01-22 15:37:00+00:00
20          security-service                      3                  13 2024-01-22 15:30:00+00:00
2          assurance-service                      2                  13 2024-01-22 15:31:00+00:00
26              user-service                      3                  13 2024-01-22 15:32:00+00:00
5             config-service                      3                  12 2024-01-22 15:30:00+00:00
17             price-service                      3                  12 2024-01-22 15:30:00+00:00
0   admin-basic-info-service                      2                  12 2024-01-22 15:32:00+00:00
6      consign-price-service                      2                  11 2024-01-22 15:34:00+00:00
7            consign-service                      2                  11 2024-01-22 15:38:00+00:00
16          preserve-service                      2                  10 2024-01-22 15:30:00+00:00,       origin_cmdb_id  breach_count           earliest_breach
9               root            18 2024-01-22 15:42:00+00:00
44  ts-order-service             9 2024-01-22 15:30:00+00:00)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np

# Reuse metric_df if present, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure timestamp_dt parsed as UTC datetimes
if 'timestamp_dt' not in metric_df.columns:
    metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Target component
component = 'ts-order-service'

# Subset full-series for the component
metric_comp_full = metric_df[metric_df['cmdb_id'] == component].copy()

# If no data for component, prepare empty outputs
if metric_comp_full.empty:
    result_df = pd.DataFrame(columns=[
        'kpi_name','p95_value','total_points_in_window','anomaly_count','earliest_anomaly_timestamp','max_value_in_window'
    ])
    distinct_kpis = pd.Series(dtype=object, name='kpi_name')
else:
    # 1) Compute global P95 per kpi_name using full available series (global threshold BEFORE filtering)
    p95_df = (
        metric_comp_full
        .groupby('kpi_name', dropna=False)['value']
        .quantile(0.95)
        .reset_index()
        .rename(columns={'value': 'p95_value'})
    )

    # 2) Filter to incident window (inclusive)
    window_start = pd.to_datetime("2024-01-22 15:26:57", utc=True)
    window_end   = pd.to_datetime("2024-01-22 15:56:57", utc=True)

    metric_window = metric_comp_full[
        (metric_comp_full['timestamp_dt'] >= window_start) &
        (metric_comp_full['timestamp_dt'] <= window_end)
    ].copy()

    if metric_window.empty:
        # No data in window
        result_df = pd.DataFrame(columns=[
            'kpi_name','p95_value','total_points_in_window','anomaly_count','earliest_anomaly_timestamp','max_value_in_window'
        ])
    else:
        # 3) Merge global p95 into window rows and compute anomaly flags
        metric_window = metric_window.merge(p95_df, on='kpi_name', how='left')
        metric_window['is_anomaly'] = metric_window['value'] >= metric_window['p95_value']

        # Aggregate per kpi_name
        def agg_fn(g):
            return pd.Series({
                'p95_value': g['p95_value'].iloc[0],
                'total_points_in_window': int(g['value'].count()),
                'anomaly_count': int(g['is_anomaly'].sum()),
                'earliest_anomaly_timestamp': (g.loc[g['is_anomaly'], 'timestamp_dt'].min() if g['is_anomaly'].any() else pd.NaT),
                'max_value_in_window': (g['value'].max() if g['value'].size>0 else np.nan)
            })

        agg = metric_window.groupby('kpi_name', dropna=False).apply(agg_fn).reset_index()

        # Sort by anomaly_count desc then max_value_in_window desc and limit to top 20
        result_df = agg.sort_values(['anomaly_count', 'max_value_in_window'], ascending=[False, False]).head(20)

    # Distinct kpi_name values observed for this component (from full series)
    distinct_kpis = pd.Series(sorted(p95_df['kpi_name'].unique()), name='kpi_name')

# Display compact outputs
result_df, distinct_kpis

```
Out[7]:
```
Summary for ts-order-service (window 2024-01-22 15:26:57 → 15:56:57 UTC)

What happened
- The following KPIs each exceeded their global P95 in the incident window (each had anomaly_count = 2 out of 25 samples):
  - mem — p95 = 282,987,900; earliest anomaly 2024-01-22T15:42:00Z; max in window ≈ 284,815,100
  - diskio — p95 = 60,471.38; earliest anomaly 2024-01-22T15:31:00Z; max in window ≈ 61,415.24
  - socket — p95 = 12.78; earliest anomaly 2024-01-22T15:43:00Z; max in window = 13.0
  - workload — p95 = 6.79; earliest anomaly 2024-01-22T15:39:00Z; max in window ≈ 7.10
  - cpu — p95 = 2.87; earliest anomaly 2024-01-22T15:36:00Z; max in window ≈ 6.37
  - latency-90 — p95 ≈ 0.0856; earliest anomaly 2024-01-22T15:36:00Z; max in window ≈ 0.1169
  - latency-50 — p95 ≈ 0.00914; earliest anomaly 2024-01-22T15:42:00Z; max in window ≈ 0.00993

Distinct KPIs observed for ts-order-service: cpu, diskio, latency-50, latency-90, mem, socket, workload.

Interpretation
- Multiple resource (CPU, memory, disk I/O, socket, workload) and latency KPIs show short-lived breaches (2 samples each out of 25), occurring at slightly different times (roughly between ~15:31 and ~15:43 UTC).
- The pattern suggests brief resource and latency spikes rather than sustained degradation. These likely reflect short traffic/load surges or transient contention rather than prolonged failures.

Recommended next steps
- Correlate these timestamps with request/trace row_count spikes and upstream origins (e.g., order/travel traffic around ~15:30 and ~15:40) to find the traffic source.
- Check instance scaling, connection limits, and any batch jobs or scheduled tasks around those times.
- Review application logs and trace latencies for the specific earliest-anomaly timestamps (15:31, 15:36, 15:39, 15:42, 15:43 UTC) to identify root cause and decide whether scaling or throttling is needed.

The original code execution output of IPython Kernel is also provided below for reference:

(     kpi_name     p95_value  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window
4         mem  2.829879e+08                      25              2  2024-01-22 15:42:00+00:00         2.848151e+08
1      diskio  6.047138e+04                      25              2  2024-01-22 15:31:00+00:00         6.141524e+04
5      socket  1.278000e+01                      25              2  2024-01-22 15:43:00+00:00         1.300000e+01
6    workload  6.791513e+00                      25              2  2024-01-22 15:39:00+00:00         7.102217e+00
0         cpu  2.871601e+00                      25              2  2024-01-22 15:36:00+00:00         6.374581e+00
3  latency-90  8.559892e-02                      25              2  2024-01-22 15:36:00+00:00         1.168986e-01
2  latency-50  9.140700e-03                      25              2  2024-01-22 15:42:00+00:00         9.933539e-03, 0           cpu
1        diskio
2    latency-50
3    latency-90
4           mem
5        socket
6      workload
Name: kpi_name, dtype: object)```
```